<a href="https://colab.research.google.com/github/charansai12322/finetuningllms/blob/main/Finetuning_LLMs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##Install Dependencies

In [ ]:
%%capture
!pip install --upgrade --no-cache-dir unsloth unsloth_zoo
!pip install --upgrade --no-deps trl peft accelerate bitsandbytes xformers

###Load the Model Using Unsloth


In [ ]:
from unsloth import FastModel

model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/gemma-3-270m-it",
    max_seq_length = 2048,
    load_in_4bit = True,
    load_in_8bit = False,
    full_finetuning = False,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


==((====))==  Unsloth 2026.3.3: Fast Gemma3 patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gemma3 won't work! Using float32.


model.safetensors:   0%|          | 0.00/393M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/236 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/233 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/670 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

##Test Model Before Fine-Tuning

In [ ]:
from transformers import TextStreamer

def do_inference(messages, max_new_tokens=128):
    _ = model.generate(
        **tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
        ).to("cuda"),
        max_new_tokens=max_new_tokens,
        temperature=1.0,
        top_p=0.95,
        top_k=64,
        streamer=TextStreamer(tokenizer, skip_prompt=True),
    )

In [ ]:
messages = [{"role": "user", "content": "Hello there."}]
do_inference(messages)

Hello! How are you?

I'm a large language model. I'm designed to be helpful and informative by the vast data I was trained on. I'm always learning and improving.

Do you have any specific questions or need help with? I'd be happy to try and answer them.

Here's a helpful tip: Ask me anything! I'm a helpful AI.
<end_of_turn>


##Apply PEFT (Configure LoRA)

In [ ]:
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers = False,
    finetune_language_layers = True,
    finetune_attention_modules = True,
    finetune_mlp_modules = True,
    r = 8,
    lora_alpha = 8,
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
    use_gradient_checkpointing = "unsloth",
)

Unsloth: Making `model.base_model.model.model` require gradients


##Load Dataset

In [ ]:
from datasets import load_dataset

dataset = load_dataset("bebechien/MobileGameNPC", "martian", split="train")

README.md:   0%|          | 0.00/141 [00:00<?, ?B/s]

martian.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/25 [00:00<?, ? examples/s]

In [ ]:
print(f"Total samples: {len(dataset)}")
print(f"Columns: {dataset.column_names}")
print(f"player: {dataset[0]['player']}")
print(f"alien:  {dataset[0]['alien']}")

Total samples: 25
Columns: ['id', 'category', 'player', 'alien']
player: Hello there.
alien:  Gree-tongs, Terran. You'z a long way from da Blue-Sphere, yez?


In [ ]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(tokenizer, chat_template="gemma-3")

In [ ]:
formatted_texts = []
for i in range(len(dataset)):
    conversation = [
        {"role": "user", "content": dataset[i]["player"]},
        {"role": "assistant", "content": dataset[i]["alien"]},
    ]
    text = tokenizer.apply_chat_template(
        conversation,
        tokenize=False,
        add_generation_prompt=False
    )
    formatted_texts.append(text)
dataset = dataset.add_column("text", formatted_texts)

In [ ]:
print("--- After conversion (what model will train on) ---")
print(dataset[0]["text"])

--- After conversion (what model will train on) ---
<bos><start_of_turn>user
Hello there.<end_of_turn>
<start_of_turn>model
Gree-tongs, Terran. You'z a long way from da Blue-Sphere, yez?<end_of_turn>



##Train and Test the Fine-Tuned Model

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 30,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 5,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
    ),
)
trainer.train()

Unsloth: Switching to float32 training since model cannot work with float16


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/25 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 25 | Num Epochs = 30 | Total steps = 120
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 1,898,496 of 269,996,672 (0.70% trained)


Step,Training Loss
5,8.077009
10,6.458200
15,5.553262
20,5.134053
25,4.617526
30,4.390448
35,4.060439
40,3.874105
45,3.575433
50,3.490857


TrainOutput(global_step=120, training_loss=3.426181681950887, metrics={'train_runtime': 648.7751, 'train_samples_per_second': 1.156, 'train_steps_per_second': 0.185, 'total_flos': 26929003196160.0, 'train_loss': 3.426181681950887})

In [ ]:
model = FastModel.for_inference(model)
do_inference([{"role": "user", "content": "Hello there."}])

Da zorp iz zappy. Iz not very... wet. Terran iz very wet. May more of you have k'tak... wet like my genezy?<end_of_turn>
